# Wikidata Ogham Sites — Interactive Map

## About this notebook

This notebook fetches Ogham-stone records from
[Wikidata](https://www.wikidata.org) and displays their find-spots on
an interactive [Leaflet](https://leafletjs.com/) map (via
[`folium`](https://python-visualization.github.io/folium/)) with an
OpenStreetMap basemap. Two views are produced: one with county-coloured
markers, and one with a **hex-binned density grid** that aggregates the
stones onto a regular hexagonal tiling. It is a companion to
`wikidata-ogham-sites.ipynb` (bar-chart overview of the same dataset)
and part of an Open Educational Resource series on knowledge graphs
and linked open data. A browser-executable variant of this notebook
is available as `wikidata-ogham-sites-map-live.qmd`.

### What you'll learn

- How to extract **geographic coordinates** from a Wikidata SPARQL
  response. Coordinates in Wikidata are stored as
  [WKT](https://en.wikipedia.org/wiki/Well-known_text_representation_of_geometry)
  `Point(lon lat)` literals — a string format that needs a small
  parser before the values can be plotted.
- How to build an **interactive web map** from Python using
  `folium`, which wraps Leaflet in a Python-friendly API and renders
  directly inside Jupyter.
- How to **hex-bin** spatial points: a widely-used GIS technique for
  summarising point distributions without the rendering fragility of
  kernel-density heatmaps.

### Data-context notes

The query in this notebook only returns records that *have*
coordinates (`?item wdt:P625 ?geo`, not `OPTIONAL`). This is the
right call for a map — records without coordinates cannot be plotted
— but it also filters out a non-trivial share of stones, depending on
how well the data is curated at the time you run it. The overview
notebook makes the opposite choice, making coordinates optional, to
give a complete picture of counts per county. The difference between
those two result sets is itself an instructive thing to look at.

A second subtlety: Wikidata stores coordinates in WGS 84
(`EPSG:4326`), the same coordinate reference system Leaflet and
OpenStreetMap expect. No reprojection is needed. If you later query
an endpoint that uses a different projection (e.g. some national
cultural-heritage registries), you will need to reproject
(`gdf.to_crs(...)` in `geopandas`).

### Tooling notes

- **`SPARQLWrapper`** to query Wikidata. The browser-executable
  variant uses `pyodide.http.pyfetch` instead.
- **`folium`** to build the interactive map. `folium` is a Python
  wrapper around Leaflet: you describe the map in Python, and the
  library emits the HTML + JavaScript Jupyter needs to render it
  inline.
- The hex grid is computed in plain Python (`math`, `collections.Counter`)
  and handed to folium as a set of `folium.Polygon` objects. No extra
  dependencies, no canvas-rendering plugin.
- We deliberately avoid `geopandas` + `contextily` for this example.
  They are excellent tools for publication-quality static maps, but
  heavier to install (GDAL, GEOS, PROJ) and not needed for this
  pattern. If you want static maps with basemaps, they are the
  right choice.

### Requirements

```bash
pip install SPARQLWrapper pandas folium
```

## Step 1 — Defining the SPARQL query

We ask Wikidata for every Ogham stone, its find-spot, its county, and
its coordinate location. The structure is almost identical to the
overview notebook, with the one difference noted above: coordinates
are required.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import folium

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
USER_AGENT = "OER-Notebook/1.0 (NFDI4Objects)"

SPARQL_QUERY = """
SELECT ?item ?itemLabel ?geo ?site ?siteLabel ?county ?countyLabel WHERE {
  ?item wdt:P31 wd:Q2016147.
  ?item wdt:P189 ?site.
  ?site wdt:P31 wd:Q72617071.
  ?item wdt:P189 ?county.
  ?county wdt:P31 wd:Q179872.
  ?item wdt:P625 ?geo.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

def fetch_wikidata(query, endpoint=SPARQL_ENDPOINT):
    """Send a SPARQL query to the given endpoint and return the raw bindings."""
    sparql = SPARQLWrapper(endpoint, agent=USER_AGENT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.queryAndConvert()["results"]["bindings"]

def parse_wkt_point(wkt):
    """Parse a WKT 'Point(lon lat)' literal into a (lat, lon) tuple."""
    if not wkt:
        return (None, None)
    try:
        coords = wkt.replace("Point(", "").replace(")", "").split()
        lon, lat = float(coords[0]), float(coords[1])
        return (lat, lon)
    except (ValueError, IndexError):
        return (None, None)

print("\u2713 Helpers defined.")

## Step 2 — Loading the data

The `parse_wkt_point` helper defined above converts each `Point(lon lat)`
literal into a `(lat, lon)` tuple. Note the ordering: WKT is
`lon lat`, but Leaflet (and most mapping libraries) expect
`lat, lon`. Getting this the wrong way round is one of the most
common mistakes when plotting spatial data — the kind of bug where
everything "works" but all your points end up in the wrong
hemisphere.

In [ ]:
results = fetch_wikidata(SPARQL_QUERY)

rows = []
for r in results:
    lat, lon = parse_wkt_point(r.get("geo", {}).get("value"))
    rows.append({
        "item":        r["item"]["value"],
        "itemLabel":   r["itemLabel"]["value"],
        "site":        r["site"]["value"],
        "siteLabel":   r["siteLabel"]["value"],
        "county":      r["county"]["value"],
        "countyLabel": r["countyLabel"]["value"],
        "latitude":    lat,
        "longitude":   lon,
    })

df = pd.DataFrame(rows).dropna(subset=["latitude", "longitude"])
print(
    f"\u2713 {len(df)} records with coordinates loaded "
    f"({df['itemLabel'].nunique()} unique Ogham stones "
    f"across {df['countyLabel'].nunique()} counties)."
)
df.head()

## Step 3a — Map with county-coloured markers

We build the first map with one colour-coded `CircleMarker` per
stone. Each county gets its own distinctive colour from a 20-colour
palette, and each marker gets a pop-up with stone details and a
link to its Wikidata entry. We group the markers by county into
separate `FeatureGroup`s, so individual counties can be toggled
on/off via the layer control in the top-right corner.

In [ ]:
assert not df.empty, "\u26a0 DataFrame is empty \u2014 please run the loading cell first."

# One colour per county, reused consistently across both maps.
palette = [
    "#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#469990", "#9A6324", "#800000",
    "#808000", "#000075", "#e6beff", "#fabed4", "#aaffc3",
    "#fffac8", "#dcbeff", "#ffd8b1", "#a9a9a9", "#ffe119",
]
counties = sorted(df["countyLabel"].unique())
county_colors = {c: palette[i % len(palette)] for i, c in enumerate(counties)}

# Centre and bounds are shared by both maps in this notebook.
centre = [df["latitude"].mean(), df["longitude"].mean()]
bounds = [[df["latitude"].min(), df["longitude"].min()],
          [df["latitude"].max(), df["longitude"].max()]]

# --- Map 1: county-coloured markers -------------------------------------
m1 = folium.Map(location=centre, zoom_start=7, tiles="OpenStreetMap")

# One FeatureGroup per county \u2014 makes each county toggleable.
county_groups = {
    c: folium.FeatureGroup(name=f"{c} ({(df['countyLabel'] == c).sum()})")
    for c in counties
}

for _, row in df.iterrows():
    popup_html = (
        f"<b>{row['itemLabel']}</b><br>"
        f"Site: {row['siteLabel']}<br>"
        f"County: {row['countyLabel']}<br>"
        f'<a href="{row["item"]}" target="_blank" rel="noopener">Wikidata entry &#8599;</a>'
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=county_colors[row["countyLabel"]],
        weight=1.5,
        fill=True,
        fill_color=county_colors[row["countyLabel"]],
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(county_groups[row["countyLabel"]])

for g in county_groups.values():
    g.add_to(m1)

folium.LayerControl(collapsed=False).add_to(m1)
m1.fit_bounds(bounds)
m1

## Step 3b — Map with hex-binned density grid

The second map shows the same data as a **hex-binned density grid**:
we tile the area with regular hexagons, count how many stones fall
into each hex, and shade each cell by its count. Empty hexagons are
not drawn, so the result is a compact "honeycomb" that makes regional
concentration immediately obvious.

### Why hex bins instead of a heatmap?

Kernel-density heatmaps (e.g. `folium.plugins.HeatMap`) are visually
attractive but they have two well-known problems in a teaching
context:

1. **They're a canvas-rendered blur.** The result depends on zoom
   level, the chosen kernel radius, and the container size; resizing
   or exporting the map can silently break the rendering. Hex bins,
   by contrast, are plain Leaflet polygons — they resize cleanly and
   behave like any other map layer.
2. **They hide the underlying counts.** A heatmap shows relative
   intensity on an arbitrary colour ramp; a hex bin shows "this
   hexagon contains *N* stones" and you can hover over it to verify.
   For an OER, that transparency matters.

Hex binning is a small but useful GIS pattern. It also matters that
hexagons tile the plane with only *three* distinct edge directions
(rectangles have two, triangles six) and that every neighbour is
exactly the same distance away — which makes them the preferred
choice for spatial aggregation in tools like
[`h3`](https://h3geo.org/), [`kepler.gl`](https://kepler.gl/), and
`geopandas` / `QGIS`.

### A small projection trick

We compute the hex grid in a trivial "equirectangular" projection:
longitude becomes `x`, and latitude becomes `y \u22c5 cos(\u03c6\u2080)` where `\u03c6\u2080`
is a reference latitude (here, the mean latitude of our data). This
makes the hexagons roughly equal-area at the latitude of interest,
which is all we need for a continental-scale visualisation. For a
global map or proper geodesy you would want `pyproj`; for Ireland at
~53\u00b0 N the shortcut is entirely adequate.

In [ ]:
import math
from collections import Counter
from branca.element import MacroElement
from jinja2 import Template

assert not df.empty, "\u26a0 DataFrame is empty \u2014 please run the loading cell first."

# ---------------------------------------------------------------------------
# 1. Hex-bin the points in Python
# ---------------------------------------------------------------------------
# HEX_SIZE_DEG is the hexagon "radius" (centre-to-corner) measured in
# degrees of longitude. Tweak this to get bigger or smaller hexes.
# ~0.12\u00b0 at 53\u00b0N corresponds to roughly 8 km, a reasonable bin size
# for an island the size of Ireland.
HEX_SIZE_DEG = 0.12
phi0 = math.radians(df["latitude"].mean())
kx = 1.0                              # x = lon
ky = 1.0 / math.cos(phi0)             # y = lat \u00b7 1/cos(phi0)

def to_xy(lat, lon):
    """Project (lat, lon) \u2192 (x, y) in locally equal-area degrees."""
    return lon * kx, lat * ky

def from_xy(x, y):
    """Inverse projection back to (lat, lon)."""
    return y / ky, x / kx

def hex_round(q, r):
    """Round fractional axial coords to the nearest hex (cube rounding)."""
    x, z = q, r
    y = -x - z
    rx, ry, rz = round(x), round(y), round(z)
    dx, dy, dz = abs(rx - x), abs(ry - y), abs(rz - z)
    if dx > dy and dx > dz:
        rx = -ry - rz
    elif dy > dz:
        ry = -rx - rz
    else:
        rz = -rx - ry
    return int(rx), int(rz)

def point_to_hex(x, y, size):
    """Flat-top hex axial coords (q, r) for a point (x, y)."""
    q = (2.0 / 3.0) * x / size
    r = (-1.0 / 3.0) * x / size + (math.sqrt(3) / 3.0) * y / size
    return hex_round(q, r)

def hex_center(q, r, size):
    """Flat-top hex centre (x, y) from axial coords."""
    x = size * 1.5 * q
    y = size * math.sqrt(3) * (r + q / 2.0)
    return x, y

def hex_corners(cx, cy, size):
    """Six corners of a flat-top hexagon centred at (cx, cy)."""
    return [
        (cx + size * math.cos(math.radians(60 * i)),
         cy + size * math.sin(math.radians(60 * i)))
        for i in range(6)
    ]

# Count stones per hex
counts = Counter()
for _, row in df.iterrows():
    x, y = to_xy(row["latitude"], row["longitude"])
    counts[point_to_hex(x, y, HEX_SIZE_DEG)] += 1

max_count = max(counts.values()) if counts else 1
ramp = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]

def bin_color(n):
    if max_count <= 1:
        return ramp[-1]
    idx = min(len(ramp) - 1, int((n - 1) / max_count * len(ramp)))
    return ramp[idx]

def legend_bins(max_n, n_bins):
    if max_n <= 1:
        return [(1, 1)]
    edges = [1 + round(i * (max_n - 1) / n_bins) for i in range(n_bins + 1)]
    bins = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1] - (1 if i < n_bins - 1 else 0)
        if hi < lo:
            hi = lo
        bins.append((lo, hi))
    return bins

# ---------------------------------------------------------------------------
# 2. Assemble the folium map
# ---------------------------------------------------------------------------
m2 = folium.Map(location=centre, zoom_start=7, tiles="OpenStreetMap")

hex_group = folium.FeatureGroup(name="Hex-binned density")
for (q, r), n in counts.items():
    cx, cy = hex_center(q, r, HEX_SIZE_DEG)
    corners_latlon = [from_xy(x, y) for x, y in hex_corners(cx, cy, HEX_SIZE_DEG)]
    folium.Polygon(
        locations=corners_latlon,
        color="#555", weight=0.7, opacity=0.8,
        fill=True, fill_color=bin_color(n), fill_opacity=0.7,
        tooltip=f"<b>{n}</b> Ogham stone{'s' if n != 1 else ''}",
    ).add_to(hex_group)
hex_group.add_to(m2)

# ---------------------------------------------------------------------------
# 3. A small custom legend (folium doesn't ship a simple list legend)
# ---------------------------------------------------------------------------
legend_rows = [
    (f"{lo}" if lo == hi else f"{lo}\u2013{hi}", ramp[i])
    for i, (lo, hi) in enumerate(legend_bins(max_count, len(ramp)))
]

class HexLegend(MacroElement):
    def __init__(self, rows):
        super().__init__()
        self._name = "HexLegend"
        self.rows = rows
        self._template = Template("""
        {% macro html(this, kwargs) %}
        <div style="position:fixed; bottom:24px; right:24px; z-index:9999;
                    background:white; padding:8px 10px; border:1px solid #ccc;
                    border-radius:4px; font:12px/1.4 sans-serif;
                    box-shadow:0 1px 3px rgba(0,0,0,0.15)">
          <div style="font-weight:600; margin-bottom:4px">Stones per hex</div>
          {% for label, color in this.rows %}
          <div style="display:flex; align-items:center; margin:2px 0">
            <span style="display:inline-block; width:14px; height:14px;
                         background:{{ color }}; border:1px solid #777;
                         margin-right:6px"></span>{{ label }}
          </div>
          {% endfor %}
        </div>
        {% endmacro %}
        """)

m2.get_root().add_child(HexLegend(legend_rows))
folium.LayerControl(collapsed=False).add_to(m2)
m2.fit_bounds(bounds)

print(f"\u2713 {len(counts)} hexagons built, max count = {max_count}.")
m2

## Step 4 — Exploring the data

The cells below are a free playground — filter the DataFrame by
county, compute per-county aggregates, or extend the query yourself.

In [ ]:
# Example: list every Ogham stone from a given county, with coordinates.
# Change the filter below to explore other counties.
county_filter = "County Kerry"

subset = (
    df[df["countyLabel"] == county_filter]
      [["itemLabel", "siteLabel", "latitude", "longitude"]]
      .drop_duplicates()
      .reset_index(drop=True)
)
print(f"Ogham stones in {county_filter}: {len(subset)}")
subset

In [ ]:
# Per-county summary: number of stones and the geographic centroid of
# their find-spots. The centroid is a quick sanity check \u2014 it should
# fall inside the county on the map.
summary = (
    df.groupby("countyLabel")
      .agg(
          n_stones=("itemLabel", "nunique"),
          n_sites=("siteLabel", "nunique"),
          mean_lat=("latitude", "mean"),
          mean_lon=("longitude", "mean"),
      )
      .sort_values("n_stones", ascending=False)
)
summary

---

*Part of an Open Educational Resource series on knowledge graphs and
linked open data, produced in the context of
[NFDI4Objects](https://www.nfdi4objects.net/).*